__This will be a walkthrough of uploading sample data to the data management system for each of the five task types.__

1. Single Label Classification
2. Muli-Label Classification
3. Object Detection
4. Semantic Segmentation
5. Instance Segmentation

Note, these are not training samples; they are examples on how to ingest data into the infrastructure described in the README. It references the expected input data format (consistent with Ground Truth output) via the manifests in the samples folder and allows you to test the infrastructure for different label types. While I did create a logging client that returns logs in pandas DataFrame format (and include an example), I highly recommend simply using Athena in the AWS console for log and table visualization. The same holds true for the step function flow to detect any failure points.

We must login to our account and gain credentials and permissions to interact with the infrastructure. Set up a user in Identity Center assigned to the account containing the CDK app, grant necessary permissions, and login via the terminal to gain temporary credentials. Ensure the profile you are using is in the local AWS config file and points to the appropriate account, role, and AWS region.

In [ ]:
# This is the name of the CDK app you deployed.
app_name = "cvdmsv1"

# This is your profile name (see the local aws config file).
profile_name = "developers_admin"

# Login to AWS. This will redirect you to a login screen or be approved if credentials are still valid from your previous login.
!aws sso login --profile {profile_name}

The following imports the programmatic API main entry point for interacting with the app. Everything is accessible through this client.

In [ ]:
# Instantiate the client.
from cvdms_platform import CvdmsApp

app = CvdmsApp(app_name=app_name,
               profile_name=profile_name)

This initiates the upload workflow for a manifest of images and labels.

In [ ]:
# manifest_path = r"samples/single_label/old.manifest"
manifest_path = r"samples/multi_label/multi_label_truth_output.manifest"
label_type = "multi-label"
job_summary = "Upload some sample COCO images for multi label label type."
data_source = "cocoval2017"

upload_info = app.start_upload_job(manifest_path,
                                   label_type,
                                   job_summary = job_summary,
                                   data_source = data_source)

Run this cell to verify the job was started successfully and retrieve the job id.

In [ ]:
job_id = upload_info.get('job_id')
upload_error = upload_info.get('error')

if job_id:
    print(f"Upload start was a success, job id is {job_id}")
else:
    print(f"Upload start was a failure, error is {upload_error}, see log file in cvdms_platform/api_logs")

Use the log interface to see the logs for this job id stored in a pandas DataFrame. It can take some time for logs to show up.

In [ ]:
log_info = app.get_logs_by_job_id(job_id)

if not log_info.get('error'):
    log_df = log_info['logs_df']
    print('First 10 logs are:')
    print(log_df.head(10))
else:
    log_retrieval_error = log_info.get('error')
    print(f'Error getting logs: {log_retrieval_error}')